# 2019 Pypot Grid

In [ ]:
import pandas as pd
import os

# Load the parquet file from previous EDA
data_folder = os.environ.get("DATA_FOLDER_PATH_IPYNB", "../../data/")
csv_name = os.environ.get("PARQUET_2019_PROCESSED_NAME", "df_500k_processed.parquet")
csv_path = os.path.join(data_folder, csv_name)

df_500k = pd.read_parquet(csv_path)
df_500k.info()
display(df_500k.head())
print(df_500k.nunique())
print(df_500k.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   DeviceId        500000 non-null  int64         
 1   StreetId        500000 non-null  int64         
 2   ArrivalTime     500000 non-null  datetime64[us]
 3   DepartureTime   500000 non-null  datetime64[us]
 4   Hour            500000 non-null  int32         
 5   WeekDay         500000 non-null  int32         
 6   Month           500000 non-null  int32         
 7   VehiclePresent  500000 non-null  bool          
dtypes: bool(1), datetime64[us](2), int32(3), int64(2)
memory usage: 21.5 MB


,DeviceId,StreetId,ArrivalTime,DepartureTime,Hour,WeekDay,Month,VehiclePresent
0,23915,123,2019-11-03 03:00:56,2019-11-03 03:07:26,3,6,11,False
1,23914,528,2019-04-16 14:14:47,2019-04-16 14:15:23,14,1,4,False
2,23913,670,2019-09-29 01:08:22,2019-09-29 01:31:41,1,6,9,False
3,23915,123,2019-06-05 18:59:17,2019-06-05 18:59:24,18,2,6,False
4,23915,123,2019-04-01 07:03:41,2019-04-01 07:14:12,7,0,4,True


DeviceId             307
StreetId              64
ArrivalTime       445527
DepartureTime     446116
Hour                  24
WeekDay                7
Month                 12
VehiclePresent         2
dtype: int64
DeviceId          0
StreetId          0
ArrivalTime       0
DepartureTime     0
Hour              0
WeekDay           0
Month             0
VehiclePresent    0
dtype: int64


## Remuestreo temporal para PyPots

In [11]:
# Time range for the dense sample
start_date = '2019-01-01'
end_date = '2019-03-31'

time_mask = (df_500k['ArrivalTime'] >= start_date) & (df_500k['DepartureTime'] <= end_date)
df_500k_dense = df_500k[time_mask].copy()
df_500k_dense.info()

print(f"Tamaño original: {df_500k_dense.shape}")
display(df_500k_dense.head())
print(df_500k_dense.isna().sum())
print(df_500k_dense.nunique())

<class 'pandas.DataFrame'>
Index: 102566 entries, 12 to 499999
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   DeviceId        102566 non-null  int64         
 1   StreetId        102566 non-null  int64         
 2   ArrivalTime     102566 non-null  datetime64[us]
 3   DepartureTime   102566 non-null  datetime64[us]
 4   Hour            102566 non-null  int32         
 5   WeekDay         102566 non-null  int32         
 6   Month           102566 non-null  int32         
 7   VehiclePresent  102566 non-null  bool          
dtypes: bool(1), datetime64[us](2), int32(3), int64(2)
memory usage: 5.2 MB
Tamaño original: (102566, 8)


,DeviceId,StreetId,ArrivalTime,DepartureTime,Hour,WeekDay,Month,VehiclePresent
12,23916,123,2019-02-20 16:53:55,2019-02-20 16:56:02,16,2,2,True
25,23914,528,2019-02-16 12:28:24,2019-02-16 12:30:47,12,5,2,True
30,23915,123,2019-01-25 04:37:49,2019-01-25 05:23:53,4,4,1,True
34,23916,123,2019-01-18 22:57:48,2019-01-18 23:01:03,22,4,1,True
36,23915,123,2019-01-24 16:57:24,2019-01-24 16:57:58,16,3,1,False


DeviceId          0
StreetId          0
ArrivalTime       0
DepartureTime     0
Hour              0
WeekDay           0
Month             0
VehiclePresent    0
dtype: int64
DeviceId             75
StreetId             36
ArrivalTime       89925
DepartureTime     90358
Hour                 24
WeekDay               7
Month                 3
VehiclePresent        2
dtype: int64


In [ ]:
import numpy as np
import pandas as pd

# Temporal line (snapshots every 15 minutes)
min_time = df_500k_dense['ArrivalTime'].min().floor('15min')
max_time = df_500k_dense['DepartureTime'].max().ceil('15min')
grid_time = pd.date_range(start=min_time, end=max_time, freq='15min')

# Grid (Sensor x Time)
sensors_unique = df_500k_dense['DeviceId'].unique()
df_left_grid = pd.MultiIndex.from_product(
    [sensors_unique, grid_time],
    names=['DeviceId', 'Timestamp']
).to_frame(index=False)

# Sort datasets for merge_asof
left_df = df_left_grid.sort_values(['Timestamp', 'DeviceId']).reset_index(drop=True)
right_df = (
    df_500k_dense[['DeviceId', 'ArrivalTime', 'DepartureTime', 'VehiclePresent', 'StreetId']]
    .sort_values(['ArrivalTime', 'DeviceId'])
    .reset_index(drop=True)
)

# Merge asof (based on the nearest previous arrival time for each sensor and timestamp)
df_500k_grid = pd.merge_asof(
    left_df,
    right_df,
    left_on='Timestamp',
    right_on='ArrivalTime',
    by='DeviceId',
    direction='backward'
 )

# Gap if timestamp is after departure time
condition_na = df_500k_grid['Timestamp'] > df_500k_grid['DepartureTime']
df_500k_grid['VehiclePresent'] = np.where(condition_na, np.nan, df_500k_grid['VehiclePresent'])

# Recalculate temporal features
df_500k_grid['Hour'] = df_500k_grid['Timestamp'].dt.hour
df_500k_grid['WeekDay'] = df_500k_grid['Timestamp'].dt.dayofweek.astype(int)
df_500k_grid['Month'] = df_500k_grid['Timestamp'].dt.month

# Clean up features for modeling
df_500k_grid = df_500k_grid.drop(columns=['ArrivalTime', 'DepartureTime'])
df_500k_grid['StreetId'] = df_500k_grid.groupby('DeviceId')['StreetId'].transform(lambda x: x.ffill().bfill()) 

print(f'Tamaño de df_500k_grid: {df_500k_grid.shape}')
print("\nValores NA en target df_500k_grid:")
print(df_500k_grid.isnull().sum())

Tamaño de df_500k_grid: (640875, 7)

Valores NA por columna en df_500k_grid:
DeviceId               0
Timestamp              0
VehiclePresent    243454
StreetId               0
Hour                   0
WeekDay                0
Month                  0
dtype: int64


### One-Hot Encoding

In [14]:
# Categorical columns to encode
categorical_cols = ['Hour', 'WeekDay', 'Month']
for col in categorical_cols:
    df_500k_grid[col] = df_500k_grid[col].astype(str)

# One-Hot Encoding
df_final = pd.get_dummies(df_500k_grid, columns=categorical_cols, dtype=int)

print(f"Nuevo tamaño del dataset: {df_final.shape}")
display(df_final.head())

Nuevo tamaño del dataset: (640875, 38)


,DeviceId,Timestamp,VehiclePresent,StreetId,Hour_0,Hour_1,Hour_10,Hour_11,Hour_12,Hour_13,...,WeekDay_0,WeekDay_1,WeekDay_2,WeekDay_3,WeekDay_4,WeekDay_5,WeekDay_6,Month_1,Month_2,Month_3
0,23914,2019-01-01,NaN,528.0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
1,23915,2019-01-01,NaN,123.0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
2,23916,2019-01-01,False,123.0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
3,23917,2019-01-01,True,894.0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
4,23918,2019-01-01,True,926.0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
